In [ ]:
RUN_TARGET = "colab"  # "colab" | "local"


## Setup Instructions

### Running on Google Colab
1. Set `RUN_TARGET = "colab"` in Cell 1
2. Runtime > Change runtime type > T4 GPU or A100
3. Run the pip-install cell — this installs the notebook dependencies used by the MTL workflow
4. Run the Drive-mount cell — approve access so checkpoints and results can be synced automatically
5. Run the runtime setup cell to download data, the shared utility module, and a fresh ESM-2 snapshot
6. Run the remaining cells normally
7. Outputs are copied to `Google Drive > My Drive > XAllergen2.0 > models/` and `results/`

### Running locally on macOS (M-series)
1. Set `RUN_TARGET = "local"` in Cell 1
2. The Colab setup cells are skipped automatically
3. MPS is used when available, otherwise CPU
4. Outputs are saved to the local `models/` and `results/` directories


In [ ]:
if RUN_TARGET == "colab":
    import subprocess as _sp
    import sys as _sys

    _sp.run(
        [
            _sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "captum",
            "transformers>=4.30.0",
            "statsmodels",
            "huggingface_hub",
        ],
        check=True,
    )
    print("Colab environment ready.")
else:
    print("Local environment detected. Skipping Colab setup.")


In [ ]:
if RUN_TARGET == "colab":
    from google.colab import drive as _drive
    from pathlib import Path

    _drive.mount("/content/drive", force_remount=False)

    DRIVE_ROOT = Path("/content/drive/MyDrive/XAllergen2.0")
    DRIVE_MODELS = DRIVE_ROOT / "models"
    DRIVE_RESULTS = DRIVE_ROOT / "results"
    DRIVE_MODELS.mkdir(parents=True, exist_ok=True)
    DRIVE_RESULTS.mkdir(parents=True, exist_ok=True)

    print(f"Google Drive mounted: {DRIVE_ROOT}")
    print(f"Models will sync to: {DRIVE_MODELS}")
    print(f"Results will sync to: {DRIVE_RESULTS}")
else:
    print("Local run: skipping Google Drive mount.")


In [ ]:
if RUN_TARGET == "colab":
    import os
    import urllib.request as _urlreq
    from pathlib import Path

    from huggingface_hub import snapshot_download

    RUNTIME_ROOT = Path("/content/XAllergen2.0")
    _DATA_DIR = RUNTIME_ROOT / "data"
    _MODEL_DIR = RUNTIME_ROOT / "models"
    _RESULTS_DIR = RUNTIME_ROOT / "results"
    _FRESH_ESM2_DIR = RUNTIME_ROOT / "hf_models" / "facebook_esm2_t6_8M_UR50D"
    for _d in [_DATA_DIR, _MODEL_DIR, _RESULTS_DIR, _FRESH_ESM2_DIR]:
        _d.mkdir(parents=True, exist_ok=True)

    _RAW = "https://raw.githubusercontent.com/Jeffateth/XAllergen2.0/main"

    _urlreq.urlretrieve(
        f"{_RAW}/baseline_notebook_utils.py",
        RUNTIME_ROOT / "baseline_notebook_utils.py",
    )
    print("Downloaded: baseline_notebook_utils.py")

    for _fname in [
        "deepalgpro_train_cleaned.csv",
        "deepalgpro_test_cleaned.csv",
        "positives_splitA.csv",
        "positives_splitB.csv",
    ]:
        _urlreq.urlretrieve(f"{_RAW}/data/{_fname}", _DATA_DIR / _fname)
        print(f"Downloaded: {_fname}")

    _fresh_path = snapshot_download(
        repo_id="facebook/esm2_t6_8M_UR50D",
        local_dir=_FRESH_ESM2_DIR,
        local_dir_use_symlinks=False,
        force_download=True,
        resume_download=False,
    )
    os.environ["XALLERGEN_HF_MODEL_DIR"] = str(_fresh_path)
    print(f"Downloaded fresh ESM-2 snapshot: {_fresh_path}")

    _baseline_summary_on_drive = DRIVE_RESULTS / "probing_summary.csv"
    if _baseline_summary_on_drive.exists():
        import shutil as _shutil

        _shutil.copy2(_baseline_summary_on_drive, _RESULTS_DIR / "probing_summary.csv")
        print(f"Copied baseline probing summary from Drive: {_baseline_summary_on_drive}")
    else:
        print("No probing_summary.csv found on Drive. Baseline-vs-MTL comparison will be skipped.")
else:
    print("Local run: skipping GitHub download and model snapshot setup.")


# 05 - Multi-Task Epitope Supervision for Attribution Alignment

This notebook extends `FrozenESMAllergenClassifier` with an auxiliary per-residue epitope head and tests whether the extra supervision improves attribution alignment against held-out IEDB B-cell epitope masks.

Training setup:
- Main task: allergenicity classification on `deepalgpro_train_cleaned.csv`
- Auxiliary task: residue-level epitope prediction on `positives_splitA.csv`
- Held-out attribution probe: `positives_splitB.csv`

Model:
- Frozen ESM-2 backbone: `esm2_t6_8M_UR50D`
- Attention pooling for sequence classification
- Two-layer MLP classifier head
- New two-layer per-residue epitope head trained with masked BCE loss


In [ ]:
import os
import sys
from pathlib import Path

if RUN_TARGET == "colab":
    RUNTIME_ROOT = Path("/content/XAllergen2.0")
    if str(RUNTIME_ROOT) not in sys.path:
        sys.path.insert(0, str(RUNTIME_ROOT))


In [ ]:
import json
import math
from pathlib import Path

from baseline_notebook_utils import (
    DROPOUT,
    ESM_MODEL_NAME,
    HF_MODEL_NAME,
    HIDDEN_DIM,
    IG_STEPS,
    MAX_SEQ_LEN,
    RANDOM_STATE,
    THRESHOLD,
    FrozenESMAllergenMTLClassifier,
    build_tokenizer,
    compute_attention_weights,
    compute_integrated_gradients,
    compute_residue_probabilities,
    configure_matplotlib_cache,
    detect_device,
    find_project_root,
    load_mtl_checkpoint,
    mean_metric_dicts,
    parse_epitope_label,
    print_runtime_context,
    seed_everything,
)

if RUN_TARGET == "colab":
    import matplotlib
    matplotlib.use("Agg")
else:
    configure_matplotlib_cache(Path.cwd())

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn.functional as F
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from torch import nn
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm


In [ ]:
# Hyperparameters
CLASSIFICATION_BATCH_SIZE = 24
EPITOPE_BATCH_SIZE        = 8
EPOCHS                    = 30
PATIENCE                  = 5
LEARNING_RATE             = 1e-3
WEIGHT_DECAY              = 1e-4
AUX_LOSS_WEIGHT           = 0.5
EPITOPE_HIDDEN_DIM        = 128
EPITOPE_VAL_FRACTION      = 0.2
N_RANDOM_DRAWS            = 100

seed_everything(RANDOM_STATE)

if RUN_TARGET == "colab":
    PROJECT_ROOT = RUNTIME_ROOT
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Device: {DEVICE}")
    if DEVICE == "cuda":
        print(f"  GPU: {torch.cuda.get_device_name(0)}")
        print(f"  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    else:
        print("  WARNING: no GPU detected - IG attribution will be slow.")
else:
    PROJECT_ROOT = find_project_root(Path.cwd())
    DEVICE = detect_device()
    print_runtime_context(DEVICE, PROJECT_ROOT)

DATA_DIR    = PROJECT_ROOT / "data"
MODEL_DIR   = PROJECT_ROOT / "models"
RESULTS_DIR = PROJECT_ROOT / "results"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_CSV            = DATA_DIR / "deepalgpro_train_cleaned.csv"
TEST_CSV             = DATA_DIR / "deepalgpro_test_cleaned.csv"
EPITOPE_TRAIN_CSV    = DATA_DIR / "positives_splitA.csv"
EPITOPE_PROBE_CSV    = DATA_DIR / "positives_splitB.csv"
CHECKPOINT_PATH      = MODEL_DIR / "mtl_frozen_esm2_epitope.pt"
METRICS_PATH         = RESULTS_DIR / "mtl_baseline_metrics.json"
PROBE_ROWS_PATH      = RESULTS_DIR / "mtl_probing_rows.csv"
PROBE_SUMMARY_PATH   = RESULTS_DIR / "mtl_probing_summary.csv"
COMPARE_SUMMARY_PATH = RESULTS_DIR / "mtl_vs_baseline_summary.csv"
PROBE_PLOT_PATH      = RESULTS_DIR / "mtl_probing_means.png"
BASELINE_SUMMARY_CSV = RESULTS_DIR / "probing_summary.csv"

tokenizer = build_tokenizer(HF_MODEL_NAME)


In [ ]:
def annotate_epitopes(frame: pd.DataFrame) -> pd.DataFrame:
    annotated = frame.copy()
    annotated["epitope_label"] = [
        parse_epitope_label(seq, start, end)
        for seq, start, end in zip(
            annotated["sequence"],
            annotated["epitope_start"],
            annotated["epitope_end"],
        )
    ]
    annotated["seq_len"] = annotated["sequence"].str.len().astype(int)
    annotated["n_epitope_residues"] = annotated["epitope_label"].map(lambda arr: int(arr.sum()))
    annotated["epitope_density"] = annotated["n_epitope_residues"] / annotated["seq_len"]
    return annotated


def filter_max_len(frame: pd.DataFrame, sequence_col: str = "sequence") -> pd.DataFrame:
    keep = frame[sequence_col].str.len() <= MAX_SEQ_LEN
    return frame.loc[keep].reset_index(drop=True)


train_df = filter_max_len(pd.read_csv(TRAIN_CSV))
test_df = filter_max_len(pd.read_csv(TEST_CSV))
epitope_train_full_df = annotate_epitopes(filter_max_len(pd.read_csv(EPITOPE_TRAIN_CSV)))
epitope_probe_df = annotate_epitopes(filter_max_len(pd.read_csv(EPITOPE_PROBE_CSV)))

train_split_df, val_split_df = train_test_split(
    train_df,
    test_size=0.1,
    stratify=train_df["label"],
    random_state=RANDOM_STATE,
)

epitope_train_df, epitope_val_df = train_test_split(
    epitope_train_full_df,
    test_size=EPITOPE_VAL_FRACTION,
    random_state=RANDOM_STATE,
)

print("DeepAlgPro train/val/test:", len(train_split_df), len(val_split_df), len(test_df))
print("Epitope train/val/probe:", len(epitope_train_df), len(epitope_val_df), len(epitope_probe_df))
print("Epitope train density mean:", round(float(epitope_train_df["epitope_density"].mean()), 4))
print("Epitope probe density mean:", round(float(epitope_probe_df["epitope_density"].mean()), 4))


In [ ]:
class ClassificationDataset(Dataset):
    def __init__(self, frame: pd.DataFrame):
        self.frame = frame.reset_index(drop=True)

    def __len__(self) -> int:
        return len(self.frame)

    def __getitem__(self, idx: int) -> dict:
        row = self.frame.iloc[idx]
        return {
            "sequence_id": row["sequence_id"],
            "sequence": row["sequence"],
            "label": int(row["label"]),
        }


class EpitopeDataset(Dataset):
    def __init__(self, frame: pd.DataFrame):
        self.frame = frame.reset_index(drop=True)

    def __len__(self) -> int:
        return len(self.frame)

    def __getitem__(self, idx: int) -> dict:
        row = self.frame.iloc[idx]
        return {
            "accession": row["accession"],
            "sequence": row["sequence"],
            "epitope_label": row["epitope_label"].astype(np.float32),
            "seq_len": int(row["seq_len"]),
        }


def collate_classification_batch(batch: list[dict]) -> dict:
    sequences = [item["sequence"] for item in batch]
    encodings = tokenizer(
        sequences,
        add_special_tokens=False,
        padding=True,
        truncation=False,
        return_tensors="pt",
    )
    return {
        "sequence_id": [item["sequence_id"] for item in batch],
        "sequence": sequences,
        "label": torch.tensor([item["label"] for item in batch], dtype=torch.float32),
        "input_ids": encodings["input_ids"],
        "attention_mask": encodings["attention_mask"],
    }


def collate_epitope_batch(batch: list[dict]) -> dict:
    sequences = [item["sequence"] for item in batch]
    encodings = tokenizer(
        sequences,
        add_special_tokens=False,
        padding=True,
        truncation=False,
        return_tensors="pt",
    )
    max_len = encodings["input_ids"].shape[1]
    labels = torch.zeros(len(batch), max_len, dtype=torch.float32)
    supervision_mask = torch.zeros(len(batch), max_len, dtype=torch.bool)

    for idx, item in enumerate(batch):
        seq_len = item["seq_len"]
        labels[idx, :seq_len] = torch.tensor(item["epitope_label"], dtype=torch.float32)
        supervision_mask[idx, :seq_len] = True

    return {
        "accession": [item["accession"] for item in batch],
        "sequence": sequences,
        "epitope_label": labels,
        "residue_supervision_mask": supervision_mask,
        "input_ids": encodings["input_ids"],
        "attention_mask": encodings["attention_mask"],
    }


def move_classification_batch_to_device(batch: dict, device: str) -> dict:
    moved = dict(batch)
    moved["input_ids"] = batch["input_ids"].to(device)
    moved["attention_mask"] = batch["attention_mask"].to(device)
    moved["label"] = batch["label"].to(device)
    return moved


def move_epitope_batch_to_device(batch: dict, device: str) -> dict:
    moved = dict(batch)
    moved["input_ids"] = batch["input_ids"].to(device)
    moved["attention_mask"] = batch["attention_mask"].to(device)
    moved["epitope_label"] = batch["epitope_label"].to(device)
    moved["residue_supervision_mask"] = batch["residue_supervision_mask"].to(device)
    return moved


train_loader = DataLoader(
    ClassificationDataset(train_split_df),
    batch_size=CLASSIFICATION_BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    collate_fn=collate_classification_batch,
)
val_loader = DataLoader(
    ClassificationDataset(val_split_df),
    batch_size=CLASSIFICATION_BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    collate_fn=collate_classification_batch,
)
test_loader = DataLoader(
    ClassificationDataset(test_df),
    batch_size=CLASSIFICATION_BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    collate_fn=collate_classification_batch,
)
epitope_train_loader = DataLoader(
    EpitopeDataset(epitope_train_df),
    batch_size=EPITOPE_BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    collate_fn=collate_epitope_batch,
)
epitope_val_loader = DataLoader(
    EpitopeDataset(epitope_val_df),
    batch_size=EPITOPE_BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    collate_fn=collate_epitope_batch,
)


In [ ]:
all_residue_labels = np.concatenate(epitope_train_df["epitope_label"].to_numpy())
positive_count = float(all_residue_labels.sum())
negative_count = float(len(all_residue_labels) - positive_count)
residue_pos_weight = torch.tensor(negative_count / max(positive_count, 1.0), device=DEVICE)

print(f"Residue positives: {positive_count:.0f}")
print(f"Residue negatives: {negative_count:.0f}")
print(f"Residue pos_weight: {float(residue_pos_weight.item()):.3f}")

model = FrozenESMAllergenMTLClassifier(
    HF_MODEL_NAME,
    hidden_dim=HIDDEN_DIM,
    dropout=DROPOUT,
    epitope_hidden_dim=EPITOPE_HIDDEN_DIM,
).to(DEVICE)

assert not any(param.requires_grad for param in model.backbone.parameters())
trainable_params = [param for param in model.parameters() if param.requires_grad]
optimizer = torch.optim.AdamW(trainable_params, lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
classification_criterion = nn.BCEWithLogitsLoss()

print(f"Trainable parameter tensors: {len(trainable_params)}")
print(f"Backbone hidden size: {model.backbone.config.hidden_size}")


In [ ]:
def compute_masked_residue_loss(
    residue_logits: torch.Tensor,
    residue_labels: torch.Tensor,
    supervision_mask: torch.Tensor,
    pos_weight: torch.Tensor,
) -> torch.Tensor:
    loss = F.binary_cross_entropy_with_logits(
        residue_logits,
        residue_labels,
        reduction="none",
        pos_weight=pos_weight,
    )
    loss = loss * supervision_mask.float()
    denom = supervision_mask.float().sum().clamp_min(1.0)
    return loss.sum() / denom


@torch.no_grad()
def evaluate_epitope_loss(
    model: nn.Module,
    loader: DataLoader,
    device: str,
    pos_weight: torch.Tensor,
) -> float:
    model.eval()
    total_loss = 0.0
    total_batches = 0
    for batch in loader:
        batch = move_epitope_batch_to_device(batch, device)
        outputs = model(batch["input_ids"], batch["attention_mask"])
        loss = compute_masked_residue_loss(
            outputs["residue_logits"],
            batch["epitope_label"],
            batch["residue_supervision_mask"],
            pos_weight=pos_weight,
        )
        total_loss += float(loss.item())
        total_batches += 1
    return total_loss / max(total_batches, 1)


def train_one_epoch_mtl(
    model: nn.Module,
    cls_loader: DataLoader,
    epi_loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    cls_criterion: nn.Module,
    device: str,
    pos_weight: torch.Tensor,
    aux_loss_weight: float,
) -> dict:
    model.train()
    total_cls_loss = 0.0
    total_epi_loss = 0.0
    total_loss = 0.0
    total_steps = 0
    epi_iter = iter(epi_loader)

    for cls_batch in cls_loader:
        cls_batch = move_classification_batch_to_device(cls_batch, device)
        try:
            epi_batch = next(epi_iter)
        except StopIteration:
            epi_iter = iter(epi_loader)
            epi_batch = next(epi_iter)
        epi_batch = move_epitope_batch_to_device(epi_batch, device)

        optimizer.zero_grad(set_to_none=True)

        cls_outputs = model(cls_batch["input_ids"], cls_batch["attention_mask"])
        cls_loss = cls_criterion(cls_outputs["logits"], cls_batch["label"])

        epi_outputs = model(epi_batch["input_ids"], epi_batch["attention_mask"])
        epi_loss = compute_masked_residue_loss(
            epi_outputs["residue_logits"],
            epi_batch["epitope_label"],
            epi_batch["residue_supervision_mask"],
            pos_weight=pos_weight,
        )

        loss = cls_loss + aux_loss_weight * epi_loss
        loss.backward()
        optimizer.step()

        total_cls_loss += float(cls_loss.item())
        total_epi_loss += float(epi_loss.item())
        total_loss += float(loss.item())
        total_steps += 1

    return {
        "train_cls_loss": total_cls_loss / max(total_steps, 1),
        "train_epi_loss": total_epi_loss / max(total_steps, 1),
        "train_total_loss": total_loss / max(total_steps, 1),
    }


@torch.no_grad()
def predict_classification(
    model: nn.Module,
    loader: DataLoader,
    device: str,
    criterion: nn.Module | None = None,
) -> tuple[float | None, pd.DataFrame]:
    model.eval()
    total_loss = 0.0
    total_examples = 0
    rows = []

    for batch in loader:
        batch = move_classification_batch_to_device(batch, device)
        outputs = model(batch["input_ids"], batch["attention_mask"])
        logits = outputs["logits"]
        probs = torch.sigmoid(logits)

        if criterion is not None:
            loss = criterion(logits, batch["label"])
            batch_size = batch["label"].shape[0]
            total_loss += float(loss.item()) * batch_size
            total_examples += batch_size

        for idx in range(len(batch["sequence_id"])):
            rows.append(
                {
                    "sequence_id": batch["sequence_id"][idx],
                    "sequence": batch["sequence"][idx],
                    "label": int(batch["label"][idx].item()),
                    "logit": float(logits[idx].item()),
                    "pred_prob": float(probs[idx].item()),
                    "pred_label": int(probs[idx].item() >= THRESHOLD),
                }
            )

    loss_value = None if criterion is None else total_loss / max(total_examples, 1)
    return loss_value, pd.DataFrame(rows)


def compute_classification_metrics(pred_df: pd.DataFrame) -> dict:
    y_true = pred_df["label"].to_numpy()
    y_prob = pred_df["pred_prob"].to_numpy()
    y_pred = pred_df["pred_label"].to_numpy()
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {
        "threshold": THRESHOLD,
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "auroc": float(roc_auc_score(y_true, y_prob)),
        "auprc": float(average_precision_score(y_true, y_prob)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "specificity": float(tn / (tn + fp)) if (tn + fp) > 0 else math.nan,
        "mcc": float(matthews_corrcoef(y_true, y_pred)),
        "confusion_matrix": {"tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp)},
    }


In [ ]:
history = []
best_val_total = float("inf")
best_epoch = 0
epochs_without_improvement = 0

for epoch in tqdm(range(1, EPOCHS + 1), desc="Training", unit="epoch"):
    train_stats = train_one_epoch_mtl(
        model,
        train_loader,
        epitope_train_loader,
        optimizer,
        classification_criterion,
        DEVICE,
        pos_weight=residue_pos_weight,
        aux_loss_weight=AUX_LOSS_WEIGHT,
    )
    val_cls_loss, val_pred_df = predict_classification(model, val_loader, DEVICE, classification_criterion)
    val_epi_loss = evaluate_epitope_loss(model, epitope_val_loader, DEVICE, residue_pos_weight)
    val_total = float(val_cls_loss) + AUX_LOSS_WEIGHT * float(val_epi_loss)

    row = {
        "epoch": epoch,
        **train_stats,
        "val_cls_loss": float(val_cls_loss),
        "val_epi_loss": float(val_epi_loss),
        "val_total_loss": float(val_total),
    }
    history.append(row)

    if val_total < best_val_total:
        best_val_total = val_total
        best_epoch = epoch
        epochs_without_improvement = 0
        torch.save(
            {
                "model_state_dict": model.state_dict(),
                "esm_model_name": ESM_MODEL_NAME,
                "architecture_hyperparameters": {
                    "hidden_dim": HIDDEN_DIM,
                    "dropout": DROPOUT,
                    "epitope_hidden_dim": EPITOPE_HIDDEN_DIM,
                },
                "training_history": history,
                "aux_loss_weight": AUX_LOSS_WEIGHT,
                "residue_pos_weight": float(residue_pos_weight.item()),
            },
            CHECKPOINT_PATH,
        )
    else:
        epochs_without_improvement += 1

    print(
        f"Epoch {epoch:>3}/{EPOCHS} | "
        f"train_cls={train_stats['train_cls_loss']:.5f} | "
        f"train_epi={train_stats['train_epi_loss']:.5f} | "
        f"val_cls={val_cls_loss:.5f} | "
        f"val_epi={val_epi_loss:.5f} | "
        f"best={best_epoch}"
    )

    if epochs_without_improvement >= PATIENCE:
        print(f"Early stopping triggered at epoch {epoch}.")
        break

print(f"Best validation objective: {best_val_total:.5f} at epoch {best_epoch}")
print(f"Checkpoint saved to: {CHECKPOINT_PATH}")


In [ ]:
model, checkpoint = load_mtl_checkpoint(
    CHECKPOINT_PATH,
    DEVICE,
    model_name=HF_MODEL_NAME,
    hidden_dim=HIDDEN_DIM,
    dropout=DROPOUT,
    epitope_hidden_dim=EPITOPE_HIDDEN_DIM,
)

history_df = pd.DataFrame(checkpoint["training_history"])
val_loss, val_predictions_df = predict_classification(model, val_loader, DEVICE, classification_criterion)
test_loss, test_predictions_df = predict_classification(model, test_loader, DEVICE, classification_criterion)
test_metrics = compute_classification_metrics(test_predictions_df)
test_metrics["test_loss"] = float(test_loss)
test_metrics["best_epoch"] = int(best_epoch)
test_metrics["n_test_sequences"] = int(len(test_predictions_df))
test_metrics["epitope_val_loss"] = float(evaluate_epitope_loss(model, epitope_val_loader, DEVICE, residue_pos_weight))

metrics_payload = {
    "esm_model_name": ESM_MODEL_NAME,
    "architecture_hyperparameters": {
        "hidden_dim": HIDDEN_DIM,
        "dropout": DROPOUT,
        "epitope_hidden_dim": EPITOPE_HIDDEN_DIM,
    },
    "training": {
        "classification_batch_size": CLASSIFICATION_BATCH_SIZE,
        "epitope_batch_size": EPITOPE_BATCH_SIZE,
        "epochs_requested": EPOCHS,
        "early_stopping_patience": PATIENCE,
        "optimizer": "AdamW",
        "lr": LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY,
        "aux_loss_weight": AUX_LOSS_WEIGHT,
        "residue_pos_weight": float(residue_pos_weight.item()),
    },
    "validation_loss": float(val_loss),
    "test_metrics": test_metrics,
}

with METRICS_PATH.open("w") as handle:
    json.dump(metrics_payload, handle, indent=2)

print(json.dumps(test_metrics, indent=2))
print(f"Saved metrics to: {METRICS_PATH}")


In [ ]:
def compute_probe_metrics(labels: np.ndarray, scores: np.ndarray) -> dict:
    labels = np.asarray(labels, dtype=np.float32)
    scores = np.asarray(scores, dtype=np.float32)
    seq_len = len(labels)
    positives = int(labels.sum())

    if seq_len == 0 or positives == 0 or positives == seq_len:
        return {"auroc": np.nan, "auprc": np.nan, "precision_at_k": np.nan}

    auroc = roc_auc_score(labels, scores)
    auprc = average_precision_score(labels, scores)

    k = max(positives, 1)
    top_k = np.argsort(scores)[-k:]
    precision_at_k = float(labels[top_k].mean())

    return {
        "auroc": float(auroc),
        "auprc": float(auprc),
        "precision_at_k": precision_at_k,
    }


rng = np.random.default_rng(RANDOM_STATE)
probe_rows = []

for _, row in tqdm(epitope_probe_df.iterrows(), total=len(epitope_probe_df), desc="Probing splitB"):
    sequence = row["sequence"]
    epitope_labels = row["epitope_label"]
    base = {
        "accession": row["accession"],
        "seq_len": int(row["seq_len"]),
        "epitope_density": float(row["epitope_density"]),
        "n_epitope_residues": int(row["n_epitope_residues"]),
    }

    residue_scores = compute_residue_probabilities(model, tokenizer, sequence, DEVICE)
    probe_rows.append({**base, "method": "residue_head", **compute_probe_metrics(epitope_labels, residue_scores)})

    attention_scores = compute_attention_weights(model, tokenizer, sequence, DEVICE)
    probe_rows.append({**base, "method": "attention_weights", **compute_probe_metrics(epitope_labels, attention_scores)})

    ig_scores = compute_integrated_gradients(
        model,
        tokenizer,
        sequence,
        DEVICE,
        steps=IG_STEPS,
        normalize=False,
    )
    probe_rows.append({**base, "method": "integrated_gradients", **compute_probe_metrics(epitope_labels, ig_scores)})

    random_metrics = [
        compute_probe_metrics(epitope_labels, rng.uniform(0.0, 1.0, size=len(epitope_labels)))
        for _ in range(N_RANDOM_DRAWS)
    ]
    probe_rows.append({**base, "method": "random_mean", **mean_metric_dicts(random_metrics)})

probe_df = pd.DataFrame(probe_rows)
probe_df.to_csv(PROBE_ROWS_PATH, index=False)
print(f"Saved row-wise probe metrics to: {PROBE_ROWS_PATH}")
probe_df.head()


In [ ]:
summary_rows = []
for method in ["residue_head", "attention_weights", "integrated_gradients", "random_mean"]:
    method_df = probe_df[probe_df["method"] == method]
    summary_rows.append(
        {
            "method": method,
            "auroc_mean": round(float(method_df["auroc"].dropna().mean()), 4),
            "auroc_sd": round(float(method_df["auroc"].dropna().std()), 4),
            "auprc_mean": round(float(method_df["auprc"].dropna().mean()), 4),
            "auprc_sd": round(float(method_df["auprc"].dropna().std()), 4),
            "precision_at_k_mean": round(float(method_df["precision_at_k"].dropna().mean()), 4),
            "precision_at_k_sd": round(float(method_df["precision_at_k"].dropna().std()), 4),
            "n_proteins": int(len(method_df)),
        }
    )

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(PROBE_SUMMARY_PATH, index=False)
print(summary_df.to_string(index=False))
print(f"Saved summary to: {PROBE_SUMMARY_PATH}")

comparison_df = None
if BASELINE_SUMMARY_CSV.exists():
    baseline_summary_df = pd.read_csv(BASELINE_SUMMARY_CSV)
    comparable_methods = ["attention_weights", "integrated_gradients", "random_mean"]
    comparison_df = baseline_summary_df[baseline_summary_df["method"].isin(comparable_methods)].merge(
        summary_df[summary_df["method"].isin(comparable_methods)],
        on="method",
        suffixes=("_baseline", "_mtl"),
    )
    for metric in ["auroc_mean", "auprc_mean", "precision_at_k_mean"]:
        comparison_df[f"delta_{metric}"] = (
            comparison_df[f"{metric}_mtl"] - comparison_df[f"{metric}_baseline"]
        ).round(4)
    comparison_df.to_csv(COMPARE_SUMMARY_PATH, index=False)
    print("
Baseline vs MTL comparison")
    print(comparison_df.to_string(index=False))
    print(f"Saved comparison to: {COMPARE_SUMMARY_PATH}")
else:
    print(f"Baseline summary not found at: {BASELINE_SUMMARY_CSV}")


In [ ]:
plot_df = summary_df.melt(
    id_vars="method",
    value_vars=["auroc_mean", "auprc_mean", "precision_at_k_mean"],
    var_name="metric",
    value_name="value",
)

metric_labels = {
    "auroc_mean": "AUROC",
    "auprc_mean": "AUPRC",
    "precision_at_k_mean": "Precision@k",
}
plot_df["metric"] = plot_df["metric"].map(metric_labels)

plt.figure(figsize=(10, 5))
sns.barplot(data=plot_df, x="metric", y="value", hue="method")
plt.ylim(0, 1)
plt.title("Held-out Epitope Alignment on positives_splitB")
plt.tight_layout()
plt.savefig(PROBE_PLOT_PATH, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved plot to: {PROBE_PLOT_PATH}")


In [ ]:
if RUN_TARGET == "colab":
    import shutil as _shutil

    DRIVE_MODELS.mkdir(parents=True, exist_ok=True)
    DRIVE_RESULTS.mkdir(parents=True, exist_ok=True)

    for _src, _dst_dir in [
        (CHECKPOINT_PATH, DRIVE_MODELS),
        (METRICS_PATH, DRIVE_RESULTS),
        (PROBE_ROWS_PATH, DRIVE_RESULTS),
        (PROBE_SUMMARY_PATH, DRIVE_RESULTS),
        (COMPARE_SUMMARY_PATH, DRIVE_RESULTS),
        (PROBE_PLOT_PATH, DRIVE_RESULTS),
    ]:
        if _src.exists():
            _shutil.copy2(_src, _dst_dir / _src.name)
            print(f"Copied to Drive: {_dst_dir / _src.name}")
        else:
            print(f"Skipped missing output: {_src}")
else:
    print("Local run: outputs saved to:")
    for _out_path in [
        CHECKPOINT_PATH,
        METRICS_PATH,
        PROBE_ROWS_PATH,
        PROBE_SUMMARY_PATH,
        COMPARE_SUMMARY_PATH,
        PROBE_PLOT_PATH,
    ]:
        print(f"  {_out_path}")


## Interpretation Guide

What to look for after running the notebook:
- `test_metrics` in `results/mtl_baseline_metrics.json`: verifies whether the auxiliary task preserves or improves sequence-level allergen classification.
- `results/mtl_probing_summary.csv`: direct held-out alignment scores for the multitask model.
- `results/mtl_vs_baseline_summary.csv`: change in attention- and IG-based alignment relative to the original baseline notebook.
- `residue_head` is the strongest sanity check: if it does not outperform random on `splitB`, the auxiliary supervision is not generalizing.
- If `residue_head` improves but attention/IG do not, the model learned epitope localization without making the classifier's own attribution more faithful.
